In [1]:
import requests

import pandas as pd

from google.cloud import bigquery

In [2]:
KEY = "579804b0fcedbcee84836facd3ac8db7"

In [4]:
from pathlib import Path

from google.oauth2 import service_account

BQ_PROJECT_ID = "gedas-analytics-dev"
BQ_DATASET_ID = "external__google_sheets"
BQ_TABLE_ID = "stocks_cost_basis"
CREDENTIALS_PATH = Path(r"d:/Coding/DAE/Airflow/credentials/gc-gedas-analytics-dev-sa.json")

credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH,
    scopes=["https://www.googleapis.com/auth/bigquery"],
)
client = bigquery.Client(project=BQ_PROJECT_ID, credentials=credentials)
symbols_df = client.query(
    f"""
    SELECT DISTINCT ticker AS symbol
    FROM `{BQ_PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}`
    WHERE ticker IS NOT NULL
    """
).to_dataframe()
symbols = symbols_df["symbol"].tolist()
symbols

c:\Users\gedim\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


['MTD',
 'TMO',
 'ADBE',
 'UNH',
 'DELL',
 'LII',
 'POOL',
 'EG',
 'META',
 'LLY',
 'HD',
 'MCD',
 'MOH',
 'MA',
 'TDY',
 'FSLR',
 'AAPL',
 'CVX',
 'PEP',
 'BIIB',
 'DOV',
 'ALB',
 'CINF',
 'GOOGL',
 'CE',
 'PKG',
 'AMZN',
 'ESS',
 'MAA',
 'DECK',
 'STLD',
 'LDOS',
 'EXR',
 'INTC',
 'WST',
 'CF',
 'NTAP',
 'OMC',
 'AKAM',
 'AVGO',
 'COF',
 'TER',
 'DG',
 'NFLX',
 'COP',
 'TRGP',
 'EXE',
 'ZBH',
 'WSM',
 'TXT',
 'COO',
 'IFF',
 'CSCO',
 'NDAQ',
 'ES',
 'LNT',
 'WRB',
 'IRM',
 'TSM',
 'BG',
 'ROL',
 'PFG',
 'VLTO',
 'BAX',
 'TSCO',
 'IP',
 'GIS',
 'SW',
 'CNP',
 'MKC',
 'FE',
 'OKE',
 'STX',
 'AMD',
 'HIMX']

In [5]:
response = requests.get(
    "https://api.marketstack.com/v2/eod/latest",
    params={
        "access_key": KEY,
        "symbols": ",".join(symbols[:3]),
        "limit": max(len(symbols), 1),
    },
)
payload = response.json()
if payload.get("error"):
    raise RuntimeError(payload["error"])

In [6]:
payload

{'pagination': {'limit': 75, 'offset': 0, 'count': 3, 'total': 3},
 'data': [{'open': 1675.0,
   'high': 1713.385,
   'low': 1666.13,
   'close': 1684.64,
   'volume': 229700.0,
   'adj_high': 1713.385,
   'adj_low': 1666.13,
   'adj_close': 1684.64,
   'adj_open': 1675.0,
   'adj_volume': 230303.0,
   'split_factor': 1.0,
   'dividend': 0.0,
   'name': 'Mettler-Toledo International Inc',
   'exchange_code': 'NYSE',
   'asset_type': 'Stock',
   'price_currency': 'USD',
   'symbol': 'MTD',
   'exchange': 'XNYS',
   'date': '2026-10-22T00:00:00+0000'},
  {'open': 691.7,
   'high': 703.2,
   'low': 674.1,
   'close': 676.54,
   'volume': 3599882.0,
   'adj_high': 703.2,
   'adj_low': 674.1,
   'adj_close': 676.54,
   'adj_open': 691.7,
   'adj_volume': 3600198.0,
   'split_factor': 1.0,
   'dividend': 0.0,
   'name': 'Thermo Fisher Scientific Inc',
   'exchange_code': 'NYSE',
   'asset_type': 'Stock',
   'price_currency': 'USD',
   'symbol': 'TMO',
   'exchange': 'XNYS',
   'date': '2026-

In [16]:
df = pd.DataFrame(payload["data"])[
    ["date", "name", "symbol", "close", "price_currency", "exchange_code"]
]
df["date"] = pd.to_datetime(df["date"], utc=True).dt.date
df["exchange_code"] = df["exchange_code"].astype(object).fillna(None)
df = df.rename(columns={"close": "close_price"})
df

,date,name,symbol,close_price,price_currency,exchange_code
0,2026-10-22,Mettler-Toledo International Inc,MTD,1684.64,USD,NYSE
1,2026-10-22,Thermo Fisher Scientific Inc,TMO,676.54,USD,NYSE
2,2026-10-22,ADOBE INC,ADBE,270.55,USD,None
